# Compute daily capacity factors for each REZ

In [190]:
client.close()
cluster.close()

INFO:dask_jobqueue.pbs:Resource specification for PBS not set, initializing it to select=1:ncpus=24:mem=90GB


In [1]:
from dask.distributed import Client,LocalCluster
from dask_jobqueue import PBSCluster

In [2]:
PROJECT = "dt6"

In [15]:
walltime = "01:00:00"
cores = 48
memory = str(4 * cores) + "GB"

cluster = PBSCluster(
    walltime=str(walltime),
    cores=cores,
    memory=str(memory),
    processes=cores,
    job_extra_directives=[
        "-q normal",
        "-P "+PROJECT,
        "-l ncpus="+str(cores),
        "-l mem="+str(memory),
        "-l storage=gdata/xp65+gdata/w42+scratch/w42+gdata/gb02+scratch/gb02+gdata/ng72+scratch/ng72+gdata/rt52"
    ],
    local_directory="$TMPDIR",
    job_directives_skip=["select"],
    log_directory="/scratch/w42/dr6273/tmp/logs"
)

INFO:dask_jobqueue.pbs:Resource specification for PBS not set, initializing it to select=1:ncpus=48:mem=179GB
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 43973 instead
  warnings.warn(


In [16]:
cluster.scale(jobs=1)
client = Client(cluster)

INFO:dask_jobqueue.pbs:Resource specification for PBS not set, initializing it to select=1:ncpus=48:mem=179GB


In [17]:
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: /proxy/43973/status,
Dashboard: /proxy/43973/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.6.121.2:42969,Workers: 0
Dashboard: /proxy/43973/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [10]:
import xarray as xr
# import numpy as np
# import pandas as pd

import glob

import matplotlib.pyplot as plt

In [6]:
%cd /g/data/w42/dr6273/work/wind_drought/
# import functions as fn

/g/data/w42/dr6273/work/wind_drought


In [7]:
# %load_ext autoreload
# %autoreload 2

## Load capacity factor data

BARRA-C2, 20min.

In [30]:
def load_barra_c2_capacity_factor():
    """
    Returns Dataset of capacity factors for 1979-2024, 20min resolution
    """
    cf_paths = sorted(glob.glob("/scratch/w42/dr6273/BARRA-C2/derived/wind_capacity_factor/*.zarr"))
    # cf_paths = cf_paths[32:-1] # select 2011-2013 files
    datasets = [xr.open_zarr(p, chunks={}) for p in cf_paths]
    return xr.combine_by_coords(datasets, compat="override", data_vars="minimal", coords="minimal")

In [31]:
cf_barraC2 = load_barra_c2_capacity_factor().cf100m

### Compute daily averages

Use `coarsen` for more efficient computation

In [36]:
cf_barraC2_daily = cf_barraC2.coarsen(time=72).mean()

In [37]:
cf_barraC2_daily["time"] = cf_barraC2_daily["time"].dt.floor("D")

In [39]:
cf_barraC2_daily = cf_barraC2_daily.chunk({"time": "300MB"})

In [41]:
cf_barraC2_daily = cf_barraC2_daily.to_dataset(name="cf100m")

In [45]:
# May or may not need
# from zarr.codecs import BloscCodec

In [48]:
cf_barraC2_daily.to_zarr(
    "/g/data/ng72/dr6273/work/projects/wind_drought/data/cf100m_vdW_BARRA-C2_20min_daily_mean_1979-2024.zarr",
    consolidated=True,
    mode="w",
    zarr_format=2 # Need this to prevent an error with zarr3: https://github.com/pydata/xarray/issues/10032
)

### Close cluster

In [55]:
client.close()
cluster.close()

INFO:dask_jobqueue.pbs:Resource specification for PBS not set, initializing it to select=1:ncpus=48:mem=179GB
